# 📘 RAG NLI Contradiction Checker
This notebook evaluates contradictions between retrieved context documents and generated answers in a Retrieval-Augmented Generation (RAG) pipeline using an NLI model (`facebook/bart-large-mnli`).

In [ ]:
# Install required libraries
!pip install transformers torch pandas

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd

In [ ]:
# Load pretrained NLI model
model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
# Define RAG context and generated answer
query = "Why was the insurance policy cancelled before expiration?"
retrieved_contexts = [
    "The policyholder requested cancellation due to a job relocation.",
    "The claim history triggered an automatic cancellation by the system.",
    "Policy cancellation happened after the insured passed away.",
    "The cancellation was due to non-payment of premiums over 3 months.",
    "There is no record of a cancellation request from the policyholder."
]
generated_answer = "The policy was cancelled because the customer missed several payments."

In [ ]:
# Function to get NLI scores
def get_nli_score(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    logits = model(**inputs).logits
    probs = F.softmax(logits, dim=1).detach().numpy().flatten()
    label_map = {0: "contradiction", 1: "neutral", 2: "entailment"}
    return label_map[probs.argmax()], probs

In [ ]:
# Run contradiction checks
results = []
for context in retrieved_contexts:
    label, scores = get_nli_score(context, generated_answer)
    results.append({
        "Context": context,
        "Label": label,
        "Contradiction Score": round(scores[0], 4),
        "Neutral Score": round(scores[1], 4),
        "Entailment Score": round(scores[2], 4)
    })
df = pd.DataFrame(results)
df

## 🔍 Additional: Prompt vs Context NLI Evaluation
Check if any retrieved context contradicts the original query (prompt) itself.

In [ ]:
# Prompt vs Context NLI checks
prompt_vs_context_results = []
for context in retrieved_contexts:
    label, scores = get_nli_score(query, context)
    prompt_vs_context_results.append({
        "Context": context,
        "Label": label,
        "Contradiction Score": round(scores[0], 4),
        "Neutral Score": round(scores[1], 4),
        "Entailment Score": round(scores[2], 4)
    })
prompt_df = pd.DataFrame(prompt_vs_context_results)
prompt_df

## 📊 Visualization
Compare contradiction scores visually for both context-to-answer and prompt-to-context evaluations.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
df['Contradiction Score'].plot(kind='bar', color='red', alpha=0.6, label='Context vs Answer')
prompt_df['Contradiction Score'].plot(kind='bar', color='blue', alpha=0.4, label='Prompt vs Context')
plt.xticks(ticks=range(len(df)), labels=[f'Ctx {i+1}' for i in range(len(df))], rotation=45)
plt.ylabel('Contradiction Score')
plt.title('Contradiction Scores (Context vs Answer and Prompt vs Context)')
plt.legend()
plt.tight_layout()
plt.show()

## 🧠 Explanation of NLI Labels and Scores

When using a Natural Language Inference (NLI) model like `facebook/bart-large-mnli`, each context and hypothesis (e.g., generated answer or prompt) pair is evaluated for semantic relationship:

| Label           | Meaning |
|------------------|--------------------------------------------------|
| **Entailment**    | The hypothesis is fully supported by the premise. |
| **Neutral**       | The relationship is uncertain but not contradictory. |
| **Contradiction** | The hypothesis logically conflicts with the premise. |

The model outputs probabilities (scores) for each of these labels. For example:

```
Contradiction: 0.70
Neutral:       0.20
Entailment:    0.10
```

In this case, the model is 70% confident the hypothesis contradicts the premise.

### 📏 Score Definitions:
- **Contradiction Score** = model's confidence that the hypothesis contradicts the premise
- **Neutral Score** = model's confidence that the hypothesis is unrelated or ambiguous
- **Entailment Score** = model's confidence that the hypothesis is supported by the premise

### ✅ How to Interpret:
| Use Case                         | Interpretation                          |
|----------------------------------|------------------------------------------|
| High Contradiction (≥ 0.7)       | Likely conflict; context may be wrong    |
| High Entailment (≥ 0.7)          | Strong alignment; context supports claim |
| High Neutral (≥ 0.7)             | Uncertain; context may be off-topic      |
| Scores ≈ uniform (~0.33 each)   | Very ambiguous or vague context          |